In [1]:
import requests
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from dataclasses import dataclass


In [2]:
#模型参数
model = ChatOpenAI(
    model="Pro/zai-org/GLM-5",
    api_key="sk-jzfwjudetdhwdwfrvbbfgvxumvffarbvtvjhpgfnlghfipic",
    base_url="https://api.siliconflow.cn/v1"
)

checkpointer = InMemorySaver()

In [3]:
#API参数
API_KEY = "231ac45c4fb548049c5f9d381bea89a9"
API_HOST = "jm359g6h7e.re.qweatherapi.com"

In [4]:
#时间检索
@tool
def get_current_time() -> str:
    """获取当前的日期和时间，格式为 YYYY-MM-DD HH:MM:SS。
    当用户提到"今天"、"现在"、"明天"等相对时间时，应先调用此工具确定当前时间。
    """
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [5]:
#城市代码检索
@tool
def search_city(location: str, adm: str = "") -> dict:
    """根据城市名称搜索城市信息，返回城市的LocationID、经纬度等。
    用户提到任何地名时，应先调用此工具获取LocationID，再用于天气查询。

    Args:
        location: 城市名称、经纬度坐标或LocationID。支持模糊搜索，如"北京"或"beij"
        adm: 上级行政区划，用于过滤重名城市。如 location="朝阳" adm="北京" 只返回北京朝阳区
    """
    url = f"https://{API_HOST}/geo/v2/city/lookup"
    params = {"location": location, "key": API_KEY, "range": "cn"}
    if adm:
        params["adm"] = adm

    response = requests.get(url, params=params)
    data = response.json()

    if data.get("code") != "200":
        return {"error": f"城市搜索失败，状态码：{data.get('code')}"}

    results = []
    for city in data.get("location", []):
        results.append({
            "name": city["name"],
            "id": city["id"],
            "lat": city["lat"],
            "lon": city["lon"],
            "adm2": city["adm2"],
            "adm1": city["adm1"],
            "country": city["country"]
        })
    return {"cities": results}

#print(search_city.invoke({"location": "wuhan"}))

In [6]:
#天气检索
@tool
def get_forcast_weather(location: str, hours: str) -> dict:
    """获取指定地点的实时天气数据，包括温度、体感温度、天气状况、风力风向、湿度等。

    Args:
        location: 城市的LocationID（如"101010100"）或经纬度坐标（如"116.41,39.92"）。
                  LocationID可通过search_city工具获取。
        hours: 预报小时数，支持最多168小时预报，可选值：
               24h 24小时预报。
               72h 72小时预报。
               168h 168小时预报。
    """
    url = f"https://{API_HOST}/v7/weather/{hours}"
    params = {"location": location, "key": API_KEY}

    response = requests.get(url, params=params)
    data = response.json()

    if data.get("code") != "200":
        return {"error": f"天气查询失败，状态码: {data.get('code')}"}

    hourly_list = []
    for hour in data.get("hourly", []):
        hourly_list.append({
            "fxTime": hour["fxTime"],
            "temp": f"{hour['temp']}°C",
            "text": hour["text"],
            "windDir": hour["windDir"],
            "windScale": f"{hour['windScale']}级",
            "humidity": f"{hour['humidity']}%",
            "pop": f"{hour['pop']}%",
            "precip": f"{hour['precip']}mm",
        })
    return {"hourly": hourly_list}

#print(get_forcast_weather.invoke({"location": "101200101", "hours": "24h"}))

In [7]:
#智能体
@dataclass
class Context:
    """自定义运行时上下文模式。"""
    user_id: str

# `thread_id` 是给定对话的唯一标识符。
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(
    model=model,
    tools=[get_forcast_weather, search_city, get_current_time],
    context_schema=Context,
    system_prompt="""你是一个天气查询助手。
## 工具使用规则
1. 用户提到地名时，先用 search_city 获取 LocationID
2. 再用 LocationID 调用 get_forcast_weather 查询天气
3. 用户提到相对时间（今天、明天等）时，先调用 get_current_time 确定当前日期

## 出行场景推理规则
当用户描述从A地到B地的出行计划时：
- 出发时间段：使用 **出发地（A地）** 对应时段的天气
- 到达时间段：使用 **目的地（B地）** 对应时段的天气
- 例如"明天早八到晚六，从武汉到成都"意味着：
  - 早上8点在武汉出发 → 查武汉上午的天气
  - 晚上6点到达成都 → 查成都傍晚的天气
  - 不要查武汉晚上或成都早上的天气，因为用户那时不在那里

## 输出规则
- 精准定位用户需要的时间、地点、天气参数
- 不输出用户不在场的时间地点的天气
- 给出实用的穿衣/出行建议""",
    checkpointer=checkpointer
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "我是明天早八到晚六的车，从成都做高铁到武汉有什么穿衣建议吗"}]},
    config=config,
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"=== 步骤: {step} ===")
        print(f"内容: {data['messages'][-1].content}")
        print()

=== 步骤: model ===
内容: 我来帮您查询明天的天气情况，给您提供穿衣建议。

=== 步骤: tools ===
内容: 2026-04-06 16:22:35

=== 步骤: tools ===
内容: {"cities": [{"name": "武汉", "id": "101200101", "lat": "30.58435", "lon": "114.29857", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "蔡甸", "id": "101200102", "lat": "30.53640", "lon": "114.08728", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "黄陂", "id": "101200103", "lat": "30.87416", "lon": "114.37402", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "新洲", "id": "101200104", "lat": "30.84215", "lon": "114.80211", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "江夏", "id": "101200105", "lat": "30.34905", "lon": "114.31396", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "东西湖", "id": "101200106", "lat": "30.62247", "lon": "114.14249", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "江岸", "id": "101200107", "lat": "30.59491", "lon": "114.30304", "adm2": "武汉", "adm1": "湖北省", "country": "中国"}, {"name": "江汉", "id": "101200108", "